In [ ]:
import numpy as np
import pandas as pd

from statsmodels.formula.api import logit, probit, ols

# Не показывать FutureWarnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# Не показывать ValueWarning, ConvergenceWarning из statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

from IPython.display import Markdown, display

# загружен датасеты в виде словаря
df_dict = {'loanapp': pd.read_csv('../datasets/loanapp.csv'), 
		'TableF5-1':pd.read_csv('../datasets/TableF5-1.csv'), 
		'SwissLabour': pd.read_csv('../datasets/SwissLabour.csv')}

## Предельные значения/эффекты

### Формулы для предельных эффектов

__для probit модели__ $P(y=1)=\Phi(x'\beta)$ формула для предельных значений

$$
\begin{aligned}
	\frac{\partial P(y=1)}{\partial x_j}&=\phi(x'\beta)\beta_j & \phi(z)=\frac{1}{\sqrt{2\pi}}exp\{-z^2/2\}
\end{aligned}
$$

__для logit модели__ $P(y=1)=\Lambda(x'\beta)$ формула для предельных значений

$$
\begin{aligned}
	\frac{\partial P(y=1)}{\partial x_j}&=\lambda(x'\beta)\beta_j & \lambda(z)=\frac{exp(z)}{(1+exp(z))^2}
\end{aligned}
$$

### Средние предельные эффекты

Обычно рассчитываются

- предельные значения для каждого регрессора в средней точке: 
  * $\phi(\bar{x}'\beta)\beta_j$ для probit 
  * $\lambda(\bar{x}'\beta)\beta_j$ для logit
- среднее по всей выборке предельное значения для каждого регрессора: 
  * $\overline{\phi(x'\beta)\beta_j}$  для probit 
  * $\overline{\lambda(x'\beta)\beta_j}$ для logit

In [ ]:
df_names = ['loanapp', 'TableF5-1', 'SwissLabour']

endog_var = ['approve', 'LFP', 'participation']

exog_vars =['I(appinc/100)+mortno+unem+dep+male',
			'WA+np.log(FAMINC)+WE+KL6+K618+CIT+UN',
			'income+age+I(age**2)+youngkids+oldkids']

mod_name = ['Approve equation', 'labour force equation', 'swiss labour force equation']

sign_level = [0.05, 0.01, 0.10]

digits = 3

for i, df_name in enumerate(df_names):
	mod_logit  = logit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	mod_probit = probit(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	mod_ols = ols(formula=endog_var[i]+'~'+exog_vars[i], data=df_dict[df_name])
	res_logit = mod_logit.fit(disp=False)
	res_probit = mod_probit.fit(disp=False)
	res_ols = mod_ols.fit(cov_type='HC3')

	display(Markdown(f'### {mod_name[i]}'))
	display(Markdown(f'Для датасет `{df_name}` оцените __probit__  и __logit__ регрессии `{endog_var[i]}` на `{exog_vars[i]}`. Для каждой регрессии вычислите предельные значения и дайте их интерпретацию:\n'
	f'- в средней точке для каждого регрессора \n'
	f'- среднее по выборке предельное значение для каждого регрессора.\n\n '
	f'Тестируйте значимость предельных эффектов при уровне значимости {sign_level[i]*100}%. Сравните с предельными эффектами в LPM-модели. __Ответ округлите до {digits} десятичных знаков__'))

	df_margeff = pd.DataFrame({'Регрессор': res_logit.params.index[1:], 
								'LPM-эффект': res_ols.params[1:].round(digits),
								'В ср.точке (logit)': res_logit.get_margeff(at='mean', dummy=True).margeff.round(digits),
								'В ср.точке (probit)': res_probit.get_margeff(at='mean', dummy=True).margeff.round(digits),
								'Ср.по выборке (logit)': res_logit.get_margeff(at='overall', dummy=True).margeff.round(digits),
								'Ср.по выборке (probit)': res_probit.get_margeff(at='overall', dummy=True).margeff.round(digits)
								})

	df_margeff_stats = pd.DataFrame({'Регрессор': res_logit.params.index[1:], 
									'LPM-эффект': res_ols.tvalues[1:].round(digits),
									'В ср.точке (logit)': res_logit.get_margeff(at='mean', dummy=True).tvalues.round(digits),
									'В ср.точке (probit)': res_probit.get_margeff(at='mean', dummy=True).tvalues.round(digits),
									'Ср.по выборке (logit)': res_logit.get_margeff(at='overall', dummy=True).tvalues.round(digits),
									'Ср.по выборке (probit)': res_probit.get_margeff(at='overall', dummy=True).tvalues.round(digits)
									})

	df_margeff_sign = pd.DataFrame({'Регрессор':res_logit.params.index[1:], 
								'LPM-эффект': res_ols.pvalues[1:].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим'),
								'В ср.точке (logit)': res_logit.get_margeff(at='mean', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим'),
								'В ср.точке (probit)': res_probit.get_margeff(at='mean', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим'),
								'Ср.по выборке (logit)': res_logit.get_margeff(at='overall', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим'),
								'Ср.по выборке (probit)': res_probit.get_margeff(at='overall', dummy=True).summary_frame()['Pr(>|z|)'].apply(lambda x: 'Значим' if x<sign_level[i] else 'Незначим')
								})

	display(Markdown(f'''__Ответ__: предельные эффекты\n'''))
	display(Markdown(df_margeff.to_markdown(index=False)))
	display(Markdown(f'''их тестовые статистики\n'''))
	display(Markdown(df_margeff_stats.to_markdown(index=False)))
	display(Markdown(f'''значимость предельных эффектов\n'''))
	display(Markdown(df_margeff_sign.to_markdown(index=False)))

